![image.png](https://i.imgur.com/a3uAqnb.png)

# Speculative Decoding from Scratch on TinyStories

- **Dataset**: [TinyStories](https://huggingface.co/datasets/roneneldan/TinyStories) from Hugging Face, short stories in simple English
- **Task**: sample from a language model faster, with the same output distribution
- **Model**: two GPT-style decoders trained from scratch, an 8-layer target (15.9M parameters) and a 1-layer draft (1.9M)
- **Runtime**: Colab T4 GPU, about 15 minutes in all

In this lab, we will:

- train a target model and a much cheaper draft
- implement speculative sampling: the draft proposes $K$ tokens, the target checks them in one call
- check that the kept tokens follow the target's distribution exactly
- predict and measure the acceptance rate and the speedup, for several $K$ and three drafts

## What is speculative decoding?

A language model writes one token per call. At batch size 1, a call scoring a few positions costs about as much as one scoring a single position: a large model spends it reading weights from memory, a small one launching GPU operations. Speculative decoding (or sampling) uses that slack: a small **draft** model proposes $K$ tokens, one call each, and the large **target** model scores them, and the position after them, in one call. Each such cycle is a **round**.

With $q$ the target's next-token distribution and $p$ the draft's, each proposal $\tilde{x}$ in turn draws its own $r$ and is accepted when

$$r < \min\left(1,\ \frac{q(\tilde{x} \mid x_{1:n})}{p(\tilde{x} \mid x_{1:n})}\right), \qquad r \sim U[0, 1]$$

The first rejected position is resampled from the **residual** $\big(q(x \mid x_{1:n}) - p(x \mid x_{1:n})\big)_+$, renormalised, and later proposals are dropped; after $K$ acceptances, one more token comes from $q$. A round yields 1 to $K + 1$ tokens, each distributed exactly as if the target had sampled it alone.

Where:
- $x_{1:n}$ is the text so far, $\tilde{x}$ a draft proposal, $r$ a uniform random number
- $(f)_+ = \max(0, f)$; renormalised means divided by its sum
- $K$ is the lookahead

Papers: [Leviathan et al., 2023](https://arxiv.org/abs/2211.17192), who swap $p$ and $q$ and write $\gamma$ for $K$, and [Chen et al., 2023](https://arxiv.org/abs/2302.01318)

## Setup

In Google Colab, select **Runtime > Change runtime type > T4 GPU**, then run the cells from top to bottom.

In [ ]:
import copy
import html
import math
import random
import time

import matplotlib.pyplot as plt
import numpy as np
from datasets import load_dataset
from huggingface_hub.utils import logging as hf_logging
from IPython.display import HTML, display
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

hf_logging.set_verbosity_error()   # the dataset is public: hide the notice that no Hugging Face token is set

In [ ]:
SEED = 42
N_TRAIN = 150_000         # training stories (the file we download holds about 498,000 plain-ASCII ones)
N_VAL = 2_000             # held-out stories: validation loss and the prompts
N_TOKENIZER = 50_000      # training stories the BPE vocabulary is learned from
VOCAB_SIZE = 4096
CONTEXT = 256             # tokens per training block, and the longest sequence a model ever sees
BATCH_SIZE = 64
TARGET_EPOCHS = 1
DRAFT_EPOCHS = 1
TARGET_LR = 1e-3          # peak learning rates
DRAFT_LR = 2e-3
WARMUP_STEPS = 100
EVAL_EVERY = 500          # optimizer steps between validation passes
NUM_WORKERS = 2           # DataLoader worker processes; a Colab runtime has 2 CPU cores

TARGET_LAYERS = 8
TARGET_DIM = 384
TARGET_HEADS = 6
DRAFT_LAYERS = 1
DRAFT_DIM = 256
DRAFT_HEADS = 4

TEMPERATURE = 1.0         # sampling temperature, applied to both models
K = 4                     # proposals per round (the lookahead)
K_VALUES = [1, 2, 4, 8]   # the lookaheads compared in section 6
N_PROMPTS = 20            # validation stories whose opening tokens are the prompts
PROMPT_TOKENS = 12        # tokens kept from each story as its prompt
MAX_NEW_TOKENS = 160      # tokens generated after each prompt, in the demos and the experiments
PROMPT = "Once upon a time, in a big"   # the prompt of the demos and of the single-position check
N_SIM = 200_000           # simulated rounds in the single-position check
N_CALLS = 2_000           # calls of speculative_generate in the same check

# every sequence stays inside the context window, so each position always sees its whole prefix
assert PROMPT_TOKENS + MAX_NEW_TOKENS + max(K_VALUES + [K]) <= CONTEXT

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")
else:
    print("No GPU found, so training will be slow and the timings will differ from a GPU's. "
          "In Colab: Runtime > Change runtime type > T4 GPU.")

# Mixed precision: float16 on a T4, which has no bfloat16 units, with a gradient scaler against
# underflow; bfloat16 on an A100 or newer, which needs no scaler; off on the CPU.
USE_AMP = DEVICE == "cuda"
AMP_DTYPE = torch.bfloat16 if USE_AMP and torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16

## 1. Dataset

- **Source**: [TinyStories](https://huggingface.co/datasets/roneneldan/TinyStories) on Hugging Face ([Eldan and Li, 2023](https://arxiv.org/abs/2305.07759))
- **Content**: about 2.1 million short stories by GPT-3.5 and GPT-4, in words a 3 to 4 year old understands; 21,990 more for validation
- **Licence**: CDLA-Sharing-1.0
- **What we use**: one training file of four (529,930 stories) and the validation file, plain-ASCII stories only, as a few percent carry mis-encoded quotes such as `â€œ` (the next cell prints the share): 150,000 to train, 2,000 to validate

In [ ]:
DATA_FILES = {
    "train": "data/train-00000-of-00004-2d5a1467fff1081b.parquet",
    "validation": "data/validation-00000-of-00001-869c898b519ad725.parquet",
}
stories = load_dataset("roneneldan/TinyStories", data_files=DATA_FILES,
                       revision="f54c09fd23315a6f9c86f9dc80f725de7d8f9c64")   # the version this lab was tested with
print(stories)


def plain_stories(split, n):
    """Up to n shuffled stories of a split that contain only ASCII characters."""
    pool = split.shuffle(seed=SEED)[:n + n // 5]["text"]             # a fifth extra, to make up for the stories we drop
    kept = [text for text in pool if text.isascii()]
    print(f"{split.split}: {1 - len(kept) / len(pool):.1%} of {len(pool):,} shuffled stories dropped as non-ASCII")
    return kept[:n]


train_texts = plain_stories(stories["train"], N_TRAIN)
val_texts = plain_stories(stories["validation"], N_VAL)
words = np.array([len(text.split()) for text in train_texts])
print(f"{len(train_texts):,} training and {len(val_texts):,} validation stories | "
      f"words per story: median {np.median(words):.0f}, 95th percentile {np.percentile(words, 95):.0f}")
print("\n" + train_texts[0])

### Tokens

A byte-level BPE vocabulary of 4,096 tokens makes most of the stories' simple words one token. The stories, each ended by `<|endoftext|>`, form one stream cut into blocks: a model reads `CONTEXT` tokens and predicts each next one. Expected time: about 2 minutes on Colab.

In [ ]:
tokenizer = Tokenizer(models.BPE())
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)   # split at spaces and punctuation, work on bytes
tokenizer.decoder = decoders.ByteLevel()
bpe_trainer = trainers.BpeTrainer(vocab_size=VOCAB_SIZE, special_tokens=["<|endoftext|>"],
                                  initial_alphabet=pre_tokenizers.ByteLevel.alphabet(), show_progress=False)
tokenizer.train_from_iterator(train_texts[:N_TOKENIZER], trainer=bpe_trainer)
EOS_ID = tokenizer.token_to_id("<|endoftext|>")
vocab_size = tokenizer.get_vocab_size()   # smaller than VOCAB_SIZE when the text has fewer pairs to merge
print(f"vocabulary: {vocab_size} tokens")
print([tokenizer.decode([i]) for i in tokenizer.encode("Once upon a time, there was a little girl named Lily.").ids])


def encode(texts, chunk=10_000):
    """All stories as one int32 array of token ids, each story followed by <|endoftext|>."""
    parts = []
    for start in tqdm(range(0, len(texts), chunk), desc="encoding", leave=False):
        for encoding in tokenizer.encode_batch(texts[start:start + chunk]):
            parts.append(np.array(encoding.ids + [EOS_ID], dtype=np.int32))
    return np.concatenate(parts)


class TokenBlocks(Dataset):
    """Block i of the token stream: inputs are tokens i*CONTEXT to (i+1)*CONTEXT - 1, targets the same span
    one token later (neighbouring blocks share one token)."""

    def __init__(self, ids):
        self.ids = torch.from_numpy(ids)

    def __len__(self):
        return (len(self.ids) - 1) // CONTEXT

    def __getitem__(self, i):
        block = self.ids[i * CONTEXT:(i + 1) * CONTEXT + 1].long()
        return block[:-1], block[1:]


train_ids, val_ids = encode(train_texts), encode(val_texts)
train_loader = DataLoader(TokenBlocks(train_ids), batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
                          num_workers=NUM_WORKERS, pin_memory=DEVICE == "cuda")
val_loader = DataLoader(TokenBlocks(val_ids), batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                        pin_memory=DEVICE == "cuda")
print(f"{len(train_ids):,} training tokens ({len(train_ids) / len(train_texts):.0f} per story), "
      f"{len(val_ids):,} validation tokens")
x, y = next(iter(train_loader))
print(x.shape, y.shape)   # (BATCH_SIZE, CONTEXT) each; y is x shifted left by one token

## 2. Model

Both models are GPT-style decoders (pre-norm blocks, causal self-attention, an output layer tied to the token embedding) sharing one tokenizer; the draft has one narrower block to the target's eight.

Real decoders keep a **KV cache**, the keys and values of earlier positions, so that a call processes only new tokens. For short code, ours rerun the whole prefix at every call; `n_last` keeps only the last `n_last` positions' logits.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.heads = heads
        self.qkv = nn.Linear(dim, 3 * dim)
        self.out = nn.Linear(dim, dim)

    def forward(self, x):                                              # x: (B, T, D)
        B, T, D = x.shape
        q, k, v = self.qkv(x).view(B, T, 3, self.heads, D // self.heads).permute(2, 0, 3, 1, 4)   # each (B, H, T, D/H)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)    # (B, H, T, D/H)
        return self.out(y.transpose(1, 2).reshape(B, T, D))            # (B, T, D)


class Block(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.ln1 = nn.LayerNorm(dim)
        self.attn = CausalSelfAttention(dim, heads)
        self.ln2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, 4 * dim), nn.GELU(), nn.Linear(4 * dim, dim))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.mlp(self.ln2(x))


class TinyGPT(nn.Module):
    def __init__(self, vocab_size, dim, layers, heads):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, dim)
        self.pos_emb = nn.Embedding(CONTEXT, dim)
        self.blocks = nn.Sequential(*[Block(dim, heads) for _ in range(layers)])
        self.ln_f = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight                         # tied input and output embeddings
        for module in self.modules():
            if isinstance(module, (nn.Linear, nn.Embedding)):
                nn.init.normal_(module.weight, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, idx, n_last=None):                               # idx: (B, T) token ids, T <= CONTEXT
        positions = torch.arange(idx.size(1), device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(positions)                # (B, T, D)
        x = self.ln_f(self.blocks(x))
        if n_last is not None:
            x = x[:, -n_last:]                                         # (B, n_last, D)
        return self.head(x)                                            # (B, T or n_last, V) logits


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())


target = TinyGPT(vocab_size, TARGET_DIM, TARGET_LAYERS, TARGET_HEADS).to(DEVICE)
draft = TinyGPT(vocab_size, DRAFT_DIM, DRAFT_LAYERS, DRAFT_HEADS).to(DEVICE)
untrained_draft = copy.deepcopy(draft).eval()                         # random weights, kept for section 7
print(f"target: {count_parameters(target):,} parameters | draft: {count_parameters(draft):,} parameters")

## 3. Training

Next-token prediction: the cross-entropy, in nats per token, between each position's logits and the next token; perplexity is $e^{\text{loss}}$. AdamW with gradient clipping; the learning rate warms up, then follows a cosine down to a tenth of its peak. Expected time: about 7 minutes on a T4.

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    """Mean next-token cross-entropy (nats per token) on a held-out loader."""
    model.eval()
    total, count = 0.0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP):
            logits = model(x)                                          # (B, CONTEXT, V)
        total += F.cross_entropy(logits.flatten(0, 1).float(), y.flatten(), reduction="sum").item()
        count += y.numel()
    return total / count


def train_one_epoch(model, optimizer, scheduler, scaler, history, name, total_steps, start):
    """One pass over the training blocks; every EVAL_EVERY steps, logs the mean training loss and the validation loss."""
    model.train()
    running_loss, running_steps = 0.0, 0
    # Pass an iterator, not the loader: given a loader, tqdm.auto calls iter() on it once more for
    # itself, which starts an extra set of worker processes every epoch that is never used or shut down.
    progress = tqdm(iter(train_loader), total=len(train_loader), desc=name, leave=False)
    for x, y in progress:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP):
            logits = model(x)                                          # (B, CONTEXT, V)
            loss = F.cross_entropy(logits.flatten(0, 1), y.flatten())
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()                                  # the scaler is a no-op when disabled
        scaler.unscale_(optimizer)                                     # clip the true gradients, not the scaled ones
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        step = scheduler.last_epoch                                    # the scheduler counts the optimizer steps
        running_loss, running_steps = running_loss + loss.item(), running_steps + 1
        progress.set_postfix(loss=f"{loss.item():.3f}")
        if step % EVAL_EVERY == 0 or step == total_steps:
            train_loss, val_loss = running_loss / running_steps, evaluate(model, val_loader)
            history["train_step"].append(step - (running_steps - 1) / 2)   # the middle of the averaged window
            running_loss, running_steps = 0.0, 0
            model.train()
            history["step"].append(step)
            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)
            print(f"{name} step {step:,}/{total_steps:,}: train loss {train_loss:.3f} | "
                  f"val loss {val_loss:.3f} (perplexity {math.exp(val_loss):.2f}) | {time.time() - start:.0f}s elapsed")


def fit(model, epochs, lr, name):
    """Trains a model for a number of epochs; returns its history of losses."""
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=0.1)
    total_steps = epochs * len(train_loader)

    def lr_factor(step):
        if step < WARMUP_STEPS:
            return (step + 1) / WARMUP_STEPS
        progress = (step - WARMUP_STEPS) / max(1, total_steps - WARMUP_STEPS)
        return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP and AMP_DTYPE == torch.float16)
    history = {"step": [], "train_step": [], "train_loss": [], "val_loss": []}
    print(f"{name}: {len(train_loader):,} optimizer steps per epoch, {epochs} epoch(s)")
    start = time.time()
    for epoch in range(1, epochs + 1):
        train_one_epoch(model, optimizer, scheduler, scaler, history, name, total_steps, start)
        print(f"{name} epoch {epoch}/{epochs} done | {time.time() - start:.0f}s elapsed")
    model.eval()
    return history


target_history = fit(target, TARGET_EPOCHS, TARGET_LR, "target")

The draft, with the higher peak learning rate usual for a smaller model. Expected time: about a minute on a T4.

In [ ]:
draft_history = fit(draft, DRAFT_EPOCHS, DRAFT_LR, "draft")

Both losses: dashed lines are the mean training loss between evaluations, plotted mid-window; solid lines the validation loss.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for name, history, color in [("target", target_history, "tab:blue"), ("draft", draft_history, "tab:orange")]:
    ax.plot(history["train_step"], history["train_loss"], "--", color=color, alpha=0.6, label=f"{name}, train")
    ax.plot(history["step"], history["val_loss"], "o-", color=color, label=f"{name}, validation")
ax.set(xlabel="optimizer step", ylabel="cross-entropy (nats per token)", title="Next-token loss")
ax.legend()
plt.tight_layout()
plt.show()

**What to notice**

The draft's loss stays above the target's: one narrow block cannot model as much of a story as eight, and wherever the two distributions differ, proposals will be rejected. After the first few hundred steps, training and validation losses coincide, since in a single epoch every batch is new.

### Plain sampling

`generate_autoregressive` samples one token per call; both models continue the same prompt.

In [ ]:
@torch.no_grad()
def next_token_probs(model, seq, n_last, temperature):
    """Next-token distributions (n_last, V) at the last n_last positions of seq (1, T)."""
    with torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP):
        logits = model(seq, n_last=n_last)                             # (1, n_last, V)
    return torch.softmax(logits[0].float() / temperature, dim=-1)


@torch.no_grad()
def generate_autoregressive(model, prompt, n_new, temperature):
    """Plain sampling, one call per new token. prompt: (1, n) token ids."""
    seq = prompt
    for _ in range(n_new):
        probs = next_token_probs(model, seq, 1, temperature)[0]        # (V,)
        seq = torch.cat([seq, torch.multinomial(probs, 1).view(1, 1)], dim=1)
    return seq, {"calls": n_new}


def story_text(seq):
    """Decodes a sequence, stopping at the first <|endoftext|>."""
    ids = seq[0].tolist()
    return tokenizer.decode(ids[:ids.index(EOS_ID)] if EOS_ID in ids else ids)


prompt = torch.tensor([tokenizer.encode(PROMPT).ids], device=DEVICE)
assert prompt.size(1) + MAX_NEW_TOKENS + K <= CONTEXT, "shorten PROMPT or MAX_NEW_TOKENS"
for name, model in [("target", target), ("draft", draft)]:
    torch.manual_seed(SEED)
    story, _ = generate_autoregressive(model, prompt, MAX_NEW_TOKENS, TEMPERATURE)
    print(f"{name}:\n{story_text(story)}\n")

## 4. Speculative Sampling

`speculative_generate` runs one round per loop iteration, its steps numbered in the comments.

In [ ]:
@torch.no_grad()
def speculative_generate(target, draft, prompt, n_new, k, temperature):
    """Speculative sampling (Chen et al., 2023, Algorithm 2) after prompt (1, n): returns the prompt plus exactly
    n_new tokens, and the counts of all rounds, including the tokens of the last round cut off at n_new."""
    seq, n_prompt = prompt, prompt.size(1)
    stats = {"target_calls": 0, "draft_calls": 0, "accepted": 0, "rejected": 0, "rounds": []}
    while seq.size(1) - n_prompt < n_new:
        # 1. the draft proposes k tokens, one call each
        proposal, draft_probs = seq, []
        for _ in range(k):
            p = next_token_probs(draft, proposal, 1, temperature)[0]   # (V,)
            proposal = torch.cat([proposal, torch.multinomial(p, 1).view(1, 1)], dim=1)
            draft_probs.append(p)
        p = torch.stack(draft_probs)                                   # (k, V): p at each proposed position
        drafted = proposal[0, -k:]                                     # (k,) the proposed tokens
        # 2. one target call: q at the k proposed positions and at the position after them
        q = next_token_probs(target, proposal, k + 1, temperature)     # (k + 1, V)
        # 3. accept proposal t when r_t < q / p; min(1, q / p) is implicit, since r_t < 1
        rows = torch.arange(k, device=q.device)
        accepted = torch.rand(k, device=q.device) < q[rows, drafted] / p[rows, drafted]
        n_accepted = int(accepted.int().cumprod(dim=0).sum())         # length of the accepted run from the start
        # 4. one token from the target: the residual after a rejection, q after k acceptances
        if n_accepted < k:
            residual = (q[n_accepted] - p[n_accepted]).clamp(min=0)
            # all zero only when p equals q up to rounding, and then q is the right distribution
            residual = torch.where(residual.sum() > 0, residual, q[n_accepted])
            next_token = torch.multinomial(residual, 1)                # multinomial renormalises its weights
            stats["rejected"] += 1
        else:
            next_token = torch.multinomial(q[k], 1)
        seq = torch.cat([seq, drafted[:n_accepted].view(1, -1), next_token.view(1, 1)], dim=1)
        stats["target_calls"] += 1
        stats["draft_calls"] += k
        stats["accepted"] += n_accepted
        stats["rounds"].append((drafted, n_accepted, next_token))
    return seq[:, :n_prompt + n_new], stats

One story, round by round: green tokens are accepted proposals, red (struck through) a rejected one, blue the target's own token, which ends each round.

In [ ]:
STYLES = {"prompt": "font-weight:bold",
          "accepted": "background:#c8e6c9;color:#000",
          "rejected": "background:#ffcdd2;color:#000;text-decoration:line-through",
          "target": "background:#bbdefb;color:#000"}


def show_rounds(prompt, rounds):
    """Shows a speculative run as coloured text, one span per group of tokens, the last round even past n_new;
    proposals dropped untested after a rejection are not drawn."""
    def span(ids, kind):
        text = html.escape(tokenizer.decode(ids, skip_special_tokens=False)).replace("\n", "<br>")   # shows <|endoftext|>
        return f'<span style="{STYLES[kind]}">{text}</span>'

    parts = [span(prompt[0].tolist(), "prompt")]
    for drafted, n_accepted, next_token in rounds:
        drafted = drafted.tolist()
        if n_accepted > 0:
            parts.append(span(drafted[:n_accepted], "accepted"))
        if n_accepted < len(drafted):
            parts.append(span(drafted[n_accepted:n_accepted + 1], "rejected"))
        parts.append(span([next_token.item()], "target"))
    display(HTML('<div style="font-family:monospace;line-height:1.9">' + "".join(parts) + "</div>"))


torch.manual_seed(SEED)
story, stats = speculative_generate(target, draft, prompt, MAX_NEW_TOKENS, K, TEMPERATURE)
show_rounds(prompt, stats["rounds"])
print(f"{stats['target_calls']} target calls for {MAX_NEW_TOKENS} tokens, {stats['draft_calls']} draft calls | "
      f"{stats['accepted']} proposals accepted, {stats['rejected']} rejected, "
      f"{stats['draft_calls'] - stats['accepted'] - stats['rejected']} dropped untested after a rejection")

**What to notice**

Most proposals the target tests are accepted, often several in a row: stock phrases such as "One day," and the ends of phrases already begun are easy. Rejections come where the story could go several ways, such as which noun or verb follows, or a name against a pronoun. The story takes far fewer target calls than tokens.

## 5. Is the Output Distribution Unchanged?

At one position, a proposal is accepted with probability $\beta = \sum_x \min\big(p(x), q(x)\big)$ (Leviathan et al., 2023, Theorem 3.5), which is $1 - \text{TV}(p, q)$: the **total variation distance** $\text{TV}(p, q) = \frac{1}{2} \sum_x |p(x) - q(x)|$ is 0 for equal distributions and 1 for disjoint ones. A token $x$ comes out accepted with probability $p(x) \min\big(1, \frac{q(x)}{p(x)}\big) = \min\big(p(x), q(x)\big)$, and from the residual, which sums to $1 - \beta$, with probability $(1 - \beta) \frac{(q(x) - p(x))_+}{1 - \beta}$; in total, $\min(p, q) + (q - p)_+ = q$.

We simulate the first token after `PROMPT`, `N_SIM` times, with the residual and with a shortcut that samples from $q$ after a rejection; `speculative_generate` then makes it `N_CALLS` times. Direct draws from $q$ measure the noise.

In [ ]:
q = next_token_probs(target, prompt, 1, TEMPERATURE)[0]                # (V,)
p = next_token_probs(draft, prompt, 1, TEMPERATURE)[0]
torch.manual_seed(SEED)
proposals = torch.multinomial(p, N_SIM, replacement=True)              # (N_SIM,) draft tokens
accepted = torch.rand(N_SIM, device=DEVICE) < q[proposals] / p[proposals]
from_residual = torch.multinomial((q - p).clamp(min=0), N_SIM, replacement=True)
from_q = torch.multinomial(q, N_SIM, replacement=True)
samples = {
    "draft tokens alone": proposals,
    "speculative, residual": torch.where(accepted, proposals, from_residual),
    "speculative, shortcut": torch.where(accepted, proposals, from_q),
    "direct draws from q": torch.multinomial(q, N_SIM, replacement=True),
}
# speculative_generate itself, asked for one token after the prompt (k = 1 is enough for the first token)
samples["speculative_generate"] = torch.cat([speculative_generate(target, draft, prompt, 1, 1, TEMPERATURE)[0][0, -1:]
                                             for _ in range(N_CALLS)])
freqs = {name: torch.bincount(tokens, minlength=vocab_size).float() / len(tokens) for name, tokens in samples.items()}


def tv_to_q(tokens):
    """Total variation distance between the token frequencies of a sample and q."""
    return 0.5 * (torch.bincount(tokens, minlength=vocab_size).float() / len(tokens) - q).abs().sum().item()


print(f"accepted: {accepted.float().mean():.4f} of proposals | sum_x min(p, q) = {torch.minimum(p, q).sum():.4f}")
for name, tokens in samples.items():
    line = f"TV({name}, q) = {tv_to_q(tokens):.4f} from {len(tokens):,} tokens"
    if len(tokens) > N_CALLS:
        line += f" | {tv_to_q(tokens[:N_CALLS]):.4f} from the first {N_CALLS:,}"
    print(line)

top = q.argsort(descending=True)[:12]
labels = [repr(tokenizer.decode([i])) for i in top.tolist()]
bars = [("q, target", q), ("p, draft", p)] + [(name, freqs[name]) for name in
        ["speculative, residual", "speculative, shortcut", "speculative_generate"]]
width = 0.8 / len(bars)
fig, ax = plt.subplots(figsize=(12, 4))
for i, (name, dist) in enumerate(bars):
    ax.bar(np.arange(len(top)) + (i - (len(bars) - 1) / 2) * width, dist[top].cpu().numpy(), width, label=name)
ax.set_xticks(np.arange(len(top)), labels, rotation=45)
ax.set(ylabel="probability", title=f"Next token after {PROMPT!r}: the 12 most likely under q")
ax.legend()
plt.tight_layout()
plt.show()

For whole stories, the target alone, speculative sampling and the draft alone each write `MAX_NEW_TOKENS` tokens after `N_PROMPTS` prompts (openings of validation stories), and the target scores each story by its mean $\log q(x_t \mid x_{<t})$. The methods take turns prompt by prompt, so their timings, used in section 6, see the same machine.

In [ ]:
def sync():
    """Waits for queued GPU work, so that the clock measures it."""
    if DEVICE == "cuda":
        torch.cuda.synchronize()


def run_methods(methods):
    """Runs every method on every prompt, the methods taking turns prompt by prompt, so that a change in the
    machine's speed affects them alike. Returns, per method, its sequences, summed counts and seconds."""
    runs = {name: {"sequences": [], "totals": {}, "seconds": 0.0} for name in methods}
    for i, prompt_ids in enumerate(prompts):
        for name, generate in methods.items():
            torch.manual_seed(SEED + i)                                # each method sees the same seed per prompt
            sync()
            start = time.perf_counter()
            seq, stats = generate(prompt_ids)
            sync()
            runs[name]["seconds"] += time.perf_counter() - start
            runs[name]["sequences"].append(seq)
            for key, value in stats.items():
                if key != "rounds":
                    runs[name]["totals"][key] = runs[name]["totals"].get(key, 0) + value
    return runs


@torch.no_grad()
def target_logprob(seq):
    """Mean log q(x_t | x_<t) over the generated tokens of seq (1, PROMPT_TOKENS + MAX_NEW_TOKENS)."""
    with torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP):
        logits = target(seq[:, :-1])                                   # position t predicts token t + 1
    log_q = torch.log_softmax(logits[0].float() / TEMPERATURE, dim=-1)
    token_logprob = log_q.gather(1, seq[0, 1:, None]).squeeze(1)
    return token_logprob[PROMPT_TOKENS - 1:].mean().item()


encoded_val = [tokenizer.encode(text).ids for text in val_texts]
prompts = [torch.tensor([ids[:PROMPT_TOKENS]], device=DEVICE) for ids in encoded_val if len(ids) >= PROMPT_TOKENS][:N_PROMPTS]
methods = {
    "target alone": lambda pr: generate_autoregressive(target, pr, MAX_NEW_TOKENS, TEMPERATURE),
    "speculative": lambda pr: speculative_generate(target, draft, pr, MAX_NEW_TOKENS, K, TEMPERATURE),
    "draft alone": lambda pr: generate_autoregressive(draft, pr, MAX_NEW_TOKENS, TEMPERATURE),
}
for generate in methods.values():                                     # warm-up: the first GPU calls are slow
    generate(prompts[0])
runs = run_methods(methods)
for name, run in runs.items():
    scores = np.array([target_logprob(seq) for seq in run["sequences"]])
    half = 2 * scores.std(ddof=1) / np.sqrt(len(scores))              # about a 95% interval for the mean
    print(f"{name:<13} mean log q per token {scores.mean():.3f}, 95% interval [{scores.mean() - half:.3f}, "
          f"{scores.mean() + half:.3f}] | {run['seconds'] / (len(prompts) * MAX_NEW_TOKENS) * 1e3:.2f} ms per token")

**What to notice**

The acceptance frequency matches $\sum_x \min(p, q)$. With the residual, the frequencies sit on $q$'s bars, within the noise of as many direct draws. The shortcut gives $\min(p, q) + (1 - \beta)\, q$, not $q$: too often wherever $p > \beta q$, too rarely elsewhere. `speculative_generate` lands close to `N_CALLS` direct draws, its bars on $q$'s where the shortcut's stray most. Whole speculative stories score like the target's own and above the draft's, a coarser check that would miss the shortcut's bias.

## 6. How Much Faster?

First, the cost of a call at several lengths. Without a KV cache, checking $K$ proposals is one call on a sequence $K$ tokens longer, so if the time barely grows with length, the check costs about one call. The **cost coefficient** $c$, a draft call's time over a target call's (Leviathan et al., 2023, Definition 3.7), comes from section 5's timings.

In [ ]:
@torch.no_grad()
def time_call(model, length, repeats=30):
    """Mean milliseconds of one call plus one sample, on a random sequence of `length` tokens."""
    seq = torch.randint(vocab_size, (1, length), device=DEVICE)
    for _ in range(3):
        next_token_probs(model, seq, 1, TEMPERATURE)
    sync()
    start = time.perf_counter()
    for _ in range(repeats):
        torch.multinomial(next_token_probs(model, seq, 1, TEMPERATURE)[0], 1)
    sync()
    return (time.perf_counter() - start) / repeats * 1e3


lengths = [16, CONTEXT // 4, CONTEXT // 2, 3 * CONTEXT // 4, CONTEXT - 16]
fig, ax = plt.subplots(figsize=(7, 4))
for name, model in [("target", target), ("draft", draft)]:
    times = [time_call(model, n) for n in lengths]
    ax.plot(lengths, times, "o-", label=name)
    print(f"{name}: " + ", ".join(f"{n} tokens {t:.2f} ms" for n, t in zip(lengths, times)))
ax.set(xlabel="sequence length (tokens)", ylabel="ms per call", title="Time of one call", ylim=(0, None))
ax.legend()
plt.tight_layout()
plt.show()

c = runs["draft alone"]["seconds"] / runs["target alone"]["seconds"]
print(f"cost coefficient c = {c:.3f}")

If each proposal were accepted independently with probability $\alpha$, then (Leviathan et al., 2023, Equation 1 and Theorem 3.8)

$$\mathbb{E}[\text{tokens per target call}] = \frac{1 - \alpha^{K+1}}{1 - \alpha}, \qquad \text{speedup} = \frac{1 - \alpha^{K+1}}{(1 - \alpha)(Kc + 1)}$$

where the speedup also pays for the $K$ draft calls per round. $\alpha$ averages $\beta$ over the positions decoding visits, so the target's own stories estimate it, as Leviathan et al. do. For each $K$ in `K_VALUES` we then measure the acceptance rate, tokens per target call and the speedup over the target alone, timed alongside. Expected time: under 2 minutes on a T4.

In [ ]:
@torch.no_grad()
def acceptance_estimate(draft_model, sequences):
    """Mean of sum_x min(p(x), q(x)) over the generated positions of sequences: the expected acceptance rate."""
    total = 0.0
    for seq in sequences:                                              # each (1, PROMPT_TOKENS + MAX_NEW_TOKENS)
        context = seq[:, :-1]                                          # position t predicts token t + 1
        q_all = next_token_probs(target, context, MAX_NEW_TOKENS, TEMPERATURE)        # (MAX_NEW_TOKENS, V)
        p_all = next_token_probs(draft_model, context, MAX_NEW_TOKENS, TEMPERATURE)
        total += torch.minimum(p_all, q_all).sum().item()
    return total / (len(sequences) * MAX_NEW_TOKENS)


def speculative(draft_model, k):
    """A method for run_methods: speculative sampling with draft_model and k proposals per round."""
    return lambda pr: speculative_generate(target, draft_model, pr, MAX_NEW_TOKENS, k, TEMPERATURE)


def summarize(results, name):
    """Acceptance rate, tokens per target call, and speedup over the target alone timed in the same results."""
    totals = results[name]["totals"]
    alpha = totals["accepted"] / max(1, totals["accepted"] + totals["rejected"])
    tokens_per_call = (totals["accepted"] + totals["target_calls"]) / totals["target_calls"]
    return alpha, tokens_per_call, results["target alone"]["seconds"] / results[name]["seconds"]


def predicted(alpha, k, c):
    """Leviathan et al. (2023), Equation 1 and Theorem 3.8."""
    tokens = (1 - alpha ** (k + 1)) / (1 - alpha) if alpha < 1 else k + 1
    return tokens, tokens / (k * c + 1)


own_stories = runs["target alone"]["sequences"]                       # the target's own samples from section 5
alpha_hat = acceptance_estimate(draft, own_stories)
print(f"acceptance rate estimated from the target's own stories: {alpha_hat:.3f}")
sweep_runs = run_methods({"target alone": methods["target alone"]} | {k: speculative(draft, k) for k in K_VALUES})
sweep = {k: summarize(sweep_runs, k) for k in K_VALUES}
print(f"{'K':>2} | {'alpha':>6} | {'tokens per target call':>23} | {'speedup':>16}")
for k, (alpha, tokens, speedup) in sweep.items():
    pred_tokens, pred_speedup = predicted(alpha_hat, k, c)
    print(f"{k:>2} | {alpha:6.3f} | {tokens:5.2f} (predicted {pred_tokens:5.2f}) | {speedup:4.2f}x (predicted {pred_speedup:4.2f}x)")

k_grid = np.arange(1, max(K_VALUES) + 1)
fig, (ax_tokens, ax_speed) = plt.subplots(1, 2, figsize=(12, 4))
ax_tokens.plot(k_grid, [predicted(alpha_hat, k, c)[0] for k in k_grid], "--", label="predicted (Equation 1)")
ax_tokens.plot(K_VALUES, [sweep[k][1] for k in K_VALUES], "o", label="measured")
ax_tokens.set(xlabel="K (proposals per round)", ylabel="tokens per target call", title="Tokens per target call")
ax_speed.plot(k_grid, [predicted(alpha_hat, k, c)[1] for k in k_grid], "--", label="predicted (Theorem 3.8)")
ax_speed.plot(K_VALUES, [sweep[k][2] for k in K_VALUES], "o", label="measured")
ax_speed.axhline(1, color="gray", lw=1)
ax_speed.set(xlabel="K (proposals per round)", ylabel="speedup over the target alone", title="Wall-clock speedup")
for ax in (ax_tokens, ax_speed):
    ax.legend()
plt.tight_layout()
plt.show()

**What to notice**

A call takes about the same time at every length: launching the GPU operations costs more than the arithmetic. $c$ still exceeds $1/8$, as both models pay alike for the output layer and the sampling. The acceptance rate matches the estimate, and tokens per call follow Equation 1. The speedup peaks at a small $K$, then falls, since every extra proposal costs a draft call. It tends to run below Theorem 3.8, which counts only model calls, not each round's work to accept, reject and resample.

## 7. What Makes a Good Draft?

We compare three drafts with `K` proposals per round: the untrained draft, the trained one, and the target as its own draft. Expected time: under 2 minutes on a T4.

In [ ]:
drafts = {"untrained draft": untrained_draft, "trained draft": draft, "target as draft": target}
draft_runs = run_methods({"target alone": methods["target alone"]} | {name: speculative(model, K) for name, model in drafts.items()})
for name, draft_model in drafts.items():
    alpha, tokens, speedup = summarize(draft_runs, name)
    print(f"{name:<15} | alpha {alpha:.3f} (estimated {acceptance_estimate(draft_model, own_stories):.3f}) | "
          f"{tokens:.2f} tokens per target call | speedup {speedup:.2f}x")

**What to notice**

The untrained draft is rejected almost every time: each round still yields one token, from the residual, but wastes $K$ draft calls; the stories remain the target's. The target as its own draft is always accepted, up to rounding, but with $c = 1$ Theorem 3.8 gives it no gain. Only the trained draft is both agreeable and cheap; its speedup differs from the sweep's at the same $K$ by timing noise alone.

## 8. Save and Reload

We save the models and the tokenizer, then reload the draft and the tokenizer and check them.

In [ ]:
torch.save({"target": target.state_dict(), "draft": draft.state_dict()}, "tinystories_target_draft.pth")
tokenizer.save("tinystories_bpe.json")

checkpoint = torch.load("tinystories_target_draft.pth", map_location=DEVICE)
reloaded = TinyGPT(vocab_size, DRAFT_DIM, DRAFT_LAYERS, DRAFT_HEADS).to(DEVICE)
reloaded.load_state_dict(checkpoint["draft"])
reloaded_tokenizer = Tokenizer.from_file("tinystories_bpe.json")
print(f"draft validation loss: {evaluate(draft, val_loader):.4f} | reloaded: {evaluate(reloaded, val_loader):.4f}")
print(f"same token ids after reloading the tokenizer: {reloaded_tokenizer.encode(PROMPT).ids == tokenizer.encode(PROMPT).ids}")

## Summary

- $K$ cheap draft calls and one target call yield 1 to $K + 1$ tokens, each on the target's distribution thanks to the $\min(1, q/p)$ test and the $(q - p)_+$ residual; the draft sets only the speed.
- The acceptance rate $\alpha$, the mean of $\sum_x \min(p, q)$ over the target's own text, can be estimated in advance.
- A speedup needs a high $\alpha$ and a small $c$; past a small $K$, more proposals cost more than they return.

## Exercises

1. **Temperature.** Put `TEMPERATURE = 0.5` first in the cell defining `speculative_generate` and run on from there. Does the acceptance rate rise or fall, and why?
2. **A deeper draft.** After the draft's training, set `DRAFT_LAYERS = 2`, train `draft = TinyGPT(vocab_size, DRAFT_DIM, DRAFT_LAYERS, DRAFT_HEADS).to(DEVICE)` with `fit`, then rerun sections 5 and 6. Which moves more, $\alpha$ or $c$, and the best speedup?
3. **Greedy decoding.** Make `speculative_generate` greedy: keep a proposal if it is the target's most likely token, else take that one. Does it match greedy plain sampling, barring near-ties?
4. **A nearly free draft.** Build a bigram draft: a buffer $P[a, b]$ on `DEVICE` holding the probability that $b$ follows $a$, counted from the token pairs of `train_ids` plus 1 each, with `forward(idx, n_last)` returning `torch.log(P[idx[:, -n_last:]])`. Add it to section 7's drafts: what $\alpha$ and speedup do you get? Leviathan et al. (2023, Section 3.6) report $\alpha \approx 0.2$ for a bigram on translation.

## Further Reading

- [Leviathan et al., 2023. Fast Inference from Transformers via Speculative Decoding](https://arxiv.org/abs/2211.17192)
- [Chen et al., 2023. Accelerating Large Language Model Decoding with Speculative Sampling](https://arxiv.org/abs/2302.01318)
- [Zhou et al., 2024. DistillSpec: Improving Speculative Decoding via Knowledge Distillation](https://arxiv.org/abs/2310.08461)
- [Cai et al., 2024. Medusa: Simple LLM Inference Acceleration Framework with Multiple Decoding Heads](https://arxiv.org/abs/2401.10774)

### Contributed by: Ali Habibullah